# 03 – Population density and population

**Purpose:** Turn SCB's population density and population per municipality 2000–2025 into an analysis-ready table. The table gets SKR's municipality groups A/B/C and the same selection of municipalities as the fertility table.

| | |
|---|---|
| **Source** | SCB, Statistikdatabasen: population density and population per municipality. SKR: *Kommungruppsindelning 2023* (municipality group classification 2023) |
| **Raw data** | `data/raw/befolkningstathet.csv`, `data/raw/Kommungruppsindelning-2023-SKR.xlsx` |
| **Helper file** | `data/processed/exkluderade_kommuner.csv` (from `02`) |
| **Output** | `data/processed/befolkningstathet.csv` |
| **Dependencies** | Run `02_FertilitetPerKommun` first. The `openpyxl` package is needed to read SKR's Excel file. |

### Decisions implemented
1. **Knivsta's zeros for 2000–2001 become `NaN`.** The municipality did not exist then, so 0 inhabitants is an error and not a measurement.
2. **Municipality group A/B/C is added** from SKR's own workbook, joined on municipality code.
3. **The same municipalities are excluded as in `02`**, i.e. those with at least 50 % missing fertility data.

## 1. Settings

Paths as in the other notebooks. 

In [1]:
from pathlib import Path

import pandas as pd
import openpyxl

# The notebook is normally run from the Assignment folder.
# If it is run from the project root, we locate Assignment anyway.
BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Cannot find data/raw. Open the notebook from the Assignment folder."


## 2. Load raw data

The file is comma-separated and encoded in **UTF-8 with BOM**, i.e. `encoding="utf-8-sig"`.

`Kommunkod` is read as **text**. Otherwise `0114` is interpreted as the number 114, and the code no longer matches other tables.

In [2]:
tathet_ra = pd.read_csv(RAW / "befolkningstathet.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})

display(tathet_ra.head())
print("Shape:", tathet_ra.shape)
print(tathet_ra.dtypes)

,Kommunkod,Kommun,Kon,År,Befolkningstäthet (invånare per kvkm),Folkmängd
0,0114,Upplands Väsby,1+2,2000,498.6,37576.0
1,0114,Upplands Väsby,1+2,2001,497.9,37524.0
2,0114,Upplands Väsby,1+2,2002,496.8,37444.0
3,0114,Upplands Väsby,1+2,2003,496.2,37397.0
4,0114,Upplands Väsby,1+2,2004,497.8,37517.0


Shape: (7540, 6)
Kommunkod                                    str
Kommun                                       str
Kon                                          str
År                                         int64
Befolkningstäthet (invånare per kvkm)    float64
Folkmängd                                float64
dtype: object


## 3. Check the structure

We expect **290 municipalities × 26 years = 7,540 rows**, four-digit municipality codes, no duplicates and no empty fields.

The column `Kon` only has the value `1+2`, i.e. both sexes. The density applies to the whole population. The column carries no information and is dropped.

In [3]:
TATHET = "Befolkningstäthet (invånare per kvkm)"

assert list(tathet_ra.columns) == ["Kommunkod", "Kommun", "Kon", "År", TATHET, "Folkmängd"]
assert tathet_ra["Kommunkod"].str.fullmatch(r"\d{4}").all()
assert tathet_ra["Kommunkod"].nunique() == 290
assert tathet_ra["År"].min() == 2000 and tathet_ra["År"].max() == 2025
assert len(tathet_ra) == 290 * 26
assert not tathet_ra.duplicated(["Kommunkod", "År"]).any()
assert tathet_ra["Kon"].unique().tolist() == ["1+2"]
assert tathet_ra.notna().all().all()

print("Density:", tathet_ra[TATHET].min(), "–", tathet_ra[TATHET].max(), "inhabitants/km²")
print("Population:", int(tathet_ra["Folkmängd"].min()), "–", int(tathet_ra["Folkmängd"].max()))

Density: 0.0 – 6529.2 inhabitants/km²
Population: 0 – 999239


## 4. Knivsta: zeros are errors, not measurements

Adding `Nan` to intial vaules of Knivsta, since it was founded from Uppsala in 2003. 

In [4]:
nollor = tathet_ra.loc[(tathet_ra[TATHET] == 0) | (tathet_ra["Folkmängd"] == 0), ["Kommun", "År", TATHET, "Folkmängd"]]
display(nollor)
assert nollor["Kommun"].eq("Knivsta").all() and nollor["År"].tolist() == [2000, 2001]

tathet = tathet_ra.copy()
tathet.loc[nollor.index, [TATHET, "Folkmängd"]] = float("nan")

uppsala = tathet_ra.loc[tathet_ra["Kommun"] == "Uppsala"].set_index("År")["Folkmängd"]
knivsta_2002 = tathet_ra.loc[(tathet_ra["Kommun"] == "Knivsta") & (tathet_ra["År"] == 2002), "Folkmängd"].iloc[0]
print(f"Uppsala 2001 → 2002: {uppsala[2002] - uppsala[2001]:+,.0f} inhabitants. Knivsta 2002: {knivsta_2002:,.0f}")

,Kommun,År,Befolkningstäthet (invånare per kvkm),Folkmängd
728,Knivsta,2000,0.0,0.0
729,Knivsta,2001,0.0,0.0


Uppsala 2001 → 2002: -11,437 inhabitants. Knivsta 2002: 12,586


## 5. Known limitation: density is rounded to one decimal

SCB reports density with **one decimal**. For the most sparsely populated municipalities this wipes out all change over time. Jokkmokk, for example, stays at 0.3 inhabitants/km² every year, even though its population fell by more than 20 %. The table shows the municipalities below 1 inhabitant/km².

A more exact density cannot be recalculated without a separate source for the municipalities' land area. It is therefore not addressed here.

In [5]:
glesa_kommuner = tathet.loc[tathet[TATHET] < 1, "Kommun"].unique()
urval = tathet.loc[tathet["Kommun"].isin(glesa_kommuner)]

glesa = urval.groupby("Kommun").agg(tathet_min=(TATHET, "min"), tathet_max=(TATHET, "max"))
folkmangd_per_ar = urval.pivot(index="Kommun", columns="År", values="Folkmängd")
glesa["folkmangd_2000"] = folkmangd_per_ar[2000]
glesa["folkmangd_2025"] = folkmangd_per_ar[2025]
glesa["folkmangd_forandring_procent"] = ((glesa["folkmangd_2025"] / glesa["folkmangd_2000"] - 1) * 100).round(0)
display(glesa)

,tathet_min,tathet_max,folkmangd_2000,folkmangd_2025,folkmangd_forandring_procent
Kommun,,,,,
Arjeplog,0.2,0.3,3384.0,2570.0,-24.0
Dorotea,0.8,1.2,3353.0,2241.0,-33.0
Härjedalen,0.9,1.0,11415.0,10117.0,-11.0
Jokkmokk,0.2,0.3,6019.0,4695.0,-22.0
Pajala,0.7,0.9,7480.0,5706.0,-24.0
Sorsele,0.3,0.4,3195.0,2362.0,-26.0
Storuman,0.8,0.9,6934.0,5543.0,-20.0
Vilhelmina,0.8,1.0,7918.0,6172.0,-22.0
Åsele,0.6,0.8,3624.0,2683.0,-26.0


## 6. Municipality groups according to SKR 2023

The municipality groups are taken directly from **SKR's workbook** (sheet *Bilaga1 Lista alla kommuner*). This is the original source.

The workbook has a heading row above the table, so `header=1` is used. The column name `" Huvudgrupp"` has a leading space in the original, so all column names are trimmed. The municipality code is read as text.

**Important about the interpretation:** A/B/C is **not** a split into small, medium and large municipalities. SKR classifies by *structure*, mainly proximity to larger cities and commuting:

| Group | Meaning | Municipalities |
|---|---|---|
| A | Large cities and municipalities near large cities | 46 |
| B | Medium-sized towns and municipalities near medium-sized towns | 110 |
| C | Smaller towns/urban areas and rural municipalities | 134 |

This is why, for example, Uppsala (about 250,000 inhabitants) is in group **B**, and the groups overlap heavily in size, as the table below shows. If the question is about size, population or density should be used.

The nine subgroups (A1–C9) are also saved for more detailed analyses. Note that the **2023** classification is used for the whole period 2000–2025. It describes the municipalities as they look today, not how they were classified at each point in time.

In [6]:
skr = pd.read_excel(
    RAW / "Kommungruppsindelning-2023-SKR.xlsx",
    sheet_name="Bilaga1 Lista alla kommuner",
    header=1,
    dtype={"Kommunkod": str},
)
skr.columns = skr.columns.str.strip()
skr = skr.dropna(subset=["Kommunkod"])
skr = skr.rename(columns={
    "Gruppkod": "kommungrupp_kod",
    "Kommunnamn": "Kommun_SKR",
    "Huvudgrupp": "kommungrupp_namn",
    "Kommungrupp 2023": "kommungrupp_undergrupp",
})
skr["kommungrupp"] = skr["kommungrupp_kod"].str[0]

assert len(skr) == 290 and not skr.duplicated("Kommunkod").any()
assert skr["Kommunkod"].str.fullmatch(r"\d{4}").all()
assert skr["kommungrupp"].value_counts().to_dict() == {"C": 134, "B": 110, "A": 46}

tathet = tathet.merge(
    skr[["Kommunkod", "Kommun_SKR", "kommungrupp", "kommungrupp_kod", "kommungrupp_namn", "kommungrupp_undergrupp"]],
    on="Kommunkod", how="left", validate="many_to_one",
)
assert tathet["kommungrupp"].notna().all(), "Some municipality code is missing from SKR's list."
assert (tathet["Kommun"] == tathet["Kommun_SKR"]).all(), "Municipality names differ between SCB and SKR."
tathet = tathet.drop(columns="Kommun_SKR")

storlek = tathet.loc[tathet["År"] == 2025].groupby("kommungrupp")["Folkmängd"].agg(["count", "min", "median", "max"])
display(storlek.astype(int))

,count,min,median,max
kommungrupp,,,,
A,46,9784,43306,999239
B,110,2327,17168,249726
C,134,2241,12706,75405


## 7. Exclude the same municipalities as in the fertility data

Municipalities with at least 50 % missing fertility data are already excluded in `02_FertilitetPerKommun`. The same municipalities are removed here. All processed municipality-level files then contain exactly the same municipalities, and a later join does not silently lose rows.

The list is read from `exkluderade_kommuner.csv` instead of being typed in by hand. If the threshold is changed in `02`, this notebook follows automatically.

In [7]:
exkl_fil = PROCESSED / "exkluderade_kommuner.csv"
assert exkl_fil.exists(), "Run 02_FertilitetPerKommun first – it creates the list of excluded municipalities."

exkluderade = pd.read_csv(exkl_fil, encoding="utf-8-sig", dtype={"Kommunkod": str})
display(exkluderade[["Kommunkod", "Kommun", "n_ar_med_data"]].merge(
    skr[["Kommunkod", "kommungrupp"]], on="Kommunkod"))

fore = tathet["Kommunkod"].nunique()
tathet = tathet.loc[~tathet["Kommunkod"].isin(exkluderade["Kommunkod"])]
print(f"Excluded {fore - tathet['Kommunkod'].nunique()} municipalities. Remaining: {tathet['Kommunkod'].nunique()}.")

,Kommunkod,Kommun,n_ar_med_data,kommungrupp
0,2403,Bjurholm,0,B
1,2425,Dorotea,2,C
2,2422,Sorsele,6,C
3,2513,Överkalix,8,C
4,2463,Åsele,12,C
5,2506,Arjeplog,12,C
6,1762,Munkfors,25,B


Excluded 7 municipalities. Remaining: 283.


## 8. Tidy columns and data types

- Column names are converted to short ASCII names (`ar`, `befolkningstathet`, `folkmangd`).
- `Kon` is dropped, since it is constant.
- Population is a number of people and is saved as an integer. 

In [8]:
tathet = (
    tathet.rename(columns={"År": "ar", TATHET: "befolkningstathet", "Folkmängd": "folkmangd"})
    [["Kommunkod", "Kommun", "ar", "befolkningstathet", "folkmangd",
      "kommungrupp", "kommungrupp_kod", "kommungrupp_namn", "kommungrupp_undergrupp"]]
    .sort_values(["Kommunkod", "ar"])
    .reset_index(drop=True)
)
assert (tathet["folkmangd"].dropna() % 1 == 0).all()
tathet["folkmangd"] = tathet["folkmangd"].round().astype("Int64")

display(tathet.head())
print(tathet.dtypes)

,Kommunkod,Kommun,ar,befolkningstathet,folkmangd,kommungrupp,kommungrupp_kod,kommungrupp_namn,kommungrupp_undergrupp
0,0114,Upplands Väsby,2000,498.6,37576,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad
1,0114,Upplands Väsby,2001,497.9,37524,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad
2,0114,Upplands Väsby,2002,496.8,37444,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad
3,0114,Upplands Väsby,2003,496.2,37397,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad
4,0114,Upplands Väsby,2004,497.8,37517,A,A2,Storstäder och storstadsnära kommuner,Pendlingskommun nära storstad


Kommunkod                     str
Kommun                        str
ar                          int64
befolkningstathet         float64
folkmangd                   Int64
kommungrupp                   str
kommungrupp_kod               str
kommungrupp_namn              str
kommungrupp_undergrupp        str
dtype: object


## 9. Final check

We check that:

- every municipality has 26 years and the key `Kommunkod` + `ar` is unique
- the only missing values are Knivsta 2000–2001
- the municipalities are **exactly the same** as in `fertilitet_kommun.csv` from `02`, so that a later join is one-to-one.

In [9]:
assert not tathet.duplicated(["Kommunkod", "ar"]).any()
assert (tathet.groupby("Kommunkod").size() == 26).all()

saknas = tathet.loc[tathet["befolkningstathet"].isna(), ["Kommun", "ar"]]
assert saknas["Kommun"].eq("Knivsta").all() and saknas["ar"].tolist() == [2000, 2001]

fert = pd.read_csv(PROCESSED / "fertilitet_kommun.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})
assert set(fert["Kommunkod"]) == set(tathet["Kommunkod"]), "The selection of municipalities differs from 02."

print(f"{tathet['Kommunkod'].nunique()} municipalities × 26 years = {len(tathet)} rows")
print("Municipalities per group:", tathet.groupby("kommungrupp")["Kommunkod"].nunique().to_dict())
print("Same municipalities as in fertilitet_kommun.csv: yes")

283 municipalities × 26 years = 7358 rows
Municipalities per group: {'A': 46, 'B': 108, 'C': 129}
Same municipalities as in fertilitet_kommun.csv: yes


## 10. Save

The municipality code is saved as text with a leading zero. Therefore read the file with `dtype={"Kommunkod": str}`.

In [10]:
utfil = PROCESSED / "befolkningstathet.csv"
tathet.to_csv(utfil, index=False, encoding="utf-8-sig")

print(f"Saved {len(tathet)} rows to {utfil.relative_to(BAS)}")

Saved 7358 rows to data/processed/befolkningstathet.csv
